# Virtual Staging Pipeline — Demo Notebook

Standalone walkthrough of the staging pipeline used by the parent web app
(see `README.md` and `cv_logic.py` for source-map details).

**This notebook contains no in-house computer-vision code** — the parent
project doesn't have any. What you'll see is:

1. PIL preprocessing (`prepare_image_for_staging`)
2. The exact prompt the production app builds (`build_staging_prompt`)
3. A call to the Replicate `bytedance/seedream-4.5` model
4. (Optional) a call to the OpenAI `gpt-image-1` fallback

The first cell sets a `USE_LIVE_API` flag — flip it to `True` once
you've set `REPLICATE_API_TOKEN` and `OPENAI_API_KEY`.

## 1. Configure the run

Set `USE_LIVE_API = True` to actually hit the APIs. Leaving it `False`
runs preprocessing locally and uses the bundled placeholder image so you
can iterate on prompts without burning credits.

In [ ]:
import os

# === Toggle this ===
USE_LIVE_API = False
# ===================

os.environ["USE_LIVE_API"] = "1" if USE_LIVE_API else "0"

if USE_LIVE_API:
    # If you keep keys in a .env file:
    #   from dotenv import load_dotenv; load_dotenv()
    assert os.getenv("REPLICATE_API_TOKEN"), "Set REPLICATE_API_TOKEN before USE_LIVE_API=True"
    assert os.getenv("OPENAI_API_KEY"),      "Set OPENAI_API_KEY before USE_LIVE_API=True"
    print("Live mode — API keys present.")
else:
    print("Offline mode — using bundled placeholder output.")

## 2. Import the pipeline

`cv_logic.py` lives next to this notebook. Importing it picks up the
`USE_LIVE_API` env var we just set.

In [ ]:
import importlib
import cv_logic
importlib.reload(cv_logic)  # in case the flag changed and the cell was re-run
from cv_logic import (
    prepare_image_for_staging,
    optimize_image_for_ai,
    build_staging_prompt,
    call_replicate_seedream,
    call_openai_fallback,
    PLACEHOLDER_INPUT,
)
print("USE_LIVE_API =", cv_logic.USE_LIVE_API)

## 3. Pick an input image

By default we use the bundled placeholder. Replace `input_path` with any
local JPG/PNG/WebP to test your own photo.

In [ ]:
from pathlib import Path
input_path = Path(PLACEHOLDER_INPUT)   # <- swap for your own photo if you like
print("Using:", input_path)

## 4. Preprocess

Resize to <= 1024 px, snap dims to multiples of 8, re-encode as JPEG q=90.
Mirrors `prepareImageForStaging` in `server/controlnet.ts`.

In [ ]:
prepared = prepare_image_for_staging(input_path, out_path="prepared.jpg")
prepared

## 5. Build the prompt

This is the same string the production web app sends to SeeDream 4.5 —
room/style sentence, then the structural-integrity constraint clause,
then the photorealism suffix.

In [ ]:
prompt = build_staging_prompt(room_type="living_room", style="modern")
print(prompt[:500], "\n...\n[", len(prompt), "chars total ]")

## 6. Call SeeDream 4.5 via Replicate

In offline mode this returns the bundled `results/placeholder_output.jpg`.
In live mode it actually hits the model.

In [ ]:
staged_seedream = call_replicate_seedream(
    prepared,
    prompt,
    out_path="staged_seedream.jpg",
)
staged_seedream

## 7. Side-by-side render

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(Image.open(input_path))
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(Image.open(staged_seedream))
axes[1].set_title("SeeDream 4.5 staged" + ("" if cv_logic.USE_LIVE_API else " (placeholder)"))
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 8. (Optional) OpenAI fallback

The production app falls back to `gpt-image-1` if Replicate is
unavailable. The fallback path uses a different preprocessing variant
(`optimize_image_for_ai` — PNG, max 1536 px).

In [ ]:
optimized = optimize_image_for_ai(input_path, out_path="optimized.png")
staged_openai = call_openai_fallback(
    optimized,
    prompt,
    out_path="staged_openai.png",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(Image.open(input_path))
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(Image.open(staged_openai))
axes[1].set_title("OpenAI gpt-image-1" + ("" if cv_logic.USE_LIVE_API else " (placeholder)"))
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 9. Iterate on the prompt

Edit the string below and re-run to see how the model responds. The
`STRUCTURAL_INTEGRITY_RULE` is what keeps walls / windows / floors
intact — try removing it to see the difference.

In [ ]:
from cv_logic import STRUCTURAL_INTEGRITY_RULE

custom_prompt = (
    "Additive virtual staging for a Modern Living Room. Place a low-profile "
    "leather sectional and matching area rug over the existing floor. "
    + STRUCTURAL_INTEGRITY_RULE
    + "\n\nPhotorealistic interior photography, perfect natural lighting, "
    "high resolution, sharp details, professional real estate photo quality."
)

custom_staged = call_replicate_seedream(
    prepared,
    custom_prompt,
    out_path="staged_custom.jpg",
)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(Image.open(custom_staged))
ax.set_title("Custom prompt run" + ("" if cv_logic.USE_LIVE_API else " (placeholder)"))
ax.axis("off")
plt.show()